[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/floating/floating.ipynb)

# Hydrostatic Stability

This example computes the hydrostatic stability of floating bodies, from small tilts to capsizing, determining whether hydrostatic pressure and gravity turn a heeled hull back upright or roll it further over. Still water presses inward on every submerged surface triangle in proportion to its depth; together those pushes carry the weight of the displaced fluid along a single upward line, and when the body heels, that line shifts sideways relative to the downward line of gravity.

In three-dimensional projective geometric algebra, the water surface is modelled as a plane, centres of gravity as points, forces as forques and turns as twists. Pressure on each wet patch acts along the normal line `samples | planes`, summed over the submerged surface into the buoyancy forque with no volume integral and added directly to the weight line `weight * (centre & (turns << -up))` to form the righting couple. Differentiating that sum across the horizontal turn space `Tilt` and pairing back gives the quadratic tilt stiffness `Tilt & (buoyancy_change + weight_change)`, whose eigenvalues divided by the weight are the roll and pitch metacentric heights.

In the Newton-Euler notation of naval architecture and hydrostatics, vertical equilibrium integrates pressure separately into the heave force $\rho g V_{\mathrm{disp}}(z_0, \phi, \theta) = W$, while rotational stability tracks the shift of the centre of buoyancy $\mathbf r_B(\phi) = V_{\mathrm{disp}}^{-1} \int_{V_{\mathrm{disp}}} \mathbf r \, \mathrm dV$ relative to the centre of gravity $\mathbf r_G$ to form the righting lever $GZ(\phi) = (y_B(\phi) - y_G)\cos\phi + (z_B(\phi) - z_G)\sin\phi$, the restoring torque $W \, GZ(\phi)$, and the upright metacentric height $GM = I_w / V_{\mathrm{disp}} - BG$ from the waterplane second moment $I_w$ and vertical separation $BG = z_G - z_B$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack, where
from numga.algebras import PGA3D as ga
from examples.fluids.floating import render, scenarios
from examples.fluids.floating.core import box_surface, wigley_hull
from examples.mechanics.simplex import simplex_inertia_weights
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                         # Scalar+
Plane = ga.gatype.vector()                                          # Vector+
Point = ga.gatype.antivector()                                      # Antivector-
Twist = ga.gatype.bivector()                                        # Bivector+
Forque = ga.gatype.antibivector()                                   # Antibivector-
Motor = ga.gatype.rotor()                                           # Rotor+
# Turns about horizontal lines through the origin: heel and trim.
Tilt = Twist.derive.nondegenerate.derive.touching(Algebra("z+"))    # Bivector+
# The energy of a small tilt.
Stiffness = ga.gatype((Scalar, Tilt, Tilt))                         # Scalar+ <- (Tilt+, Tilt+)

# Unit scalar, still water surface, upward direction, vertical shift, heel axis, and origin.
unit = mv.z | mv.z                                                  # [] Scalar+
water = mv.z                                                        # [] Plane+
up = water.dual()                                                   # [] Point-
heave = mv.w ^ water                                                # [] Twist+
heel_axis = mv.yz                                                   # [] Twist+
origin = mv.point([0, 0, 0])                                        # [] Point-
# Per triangle, the points halfway between each corner and the centroid, each standing for a third of its area.
quadrature_rule = 3 * simplex_inertia_weights(3)

fluid_weight = scenarios.fluid_weight
gravity = scenarios.gravity

## Hydrostatic push on a submerged surface

Still water presses inward on every submerged patch of a closed triangulated hull in proportion to its depth. In projective geometric algebra, the inward push on a patch is the normal line `point | plane` weighted by fluid pressure and patch area, and summing those forques over the wet surface yields the net buoyancy line without integrating over the interior volume.

The hull below is a Wigley hull with parabolic underwater sections, vertical topsides and a flat deck. Each surface triangle is clipped against the water plane into two oriented submerged triangles, and the linear depth pressure over each wet triangle is integrated exactly at three interior quadrature points, using the second-moment rule from [simplex.py](../../mechanics/simplex.py).

In [ ]:
length, beam, draft, freeboard = scenarios.length, scenarios.beam, scenarios.draft, scenarios.freeboard
n_cells = scenarios.n_cells

In [ ]:
def wet_triangles(triangles: Point, surface: Plane) -> Point:
    """The submerged part of each surface triangle below each water plane, split into two oriented triangles
    by fanning the deepest corner to the waterline crossings."""
    depths = -(surface[..., None] & triangles)                      # [..., 3] Scalar+[F]
    first_depth, second_depth, third_depth = depths[..., 0], depths[..., 1], depths[..., 2]
    deep, middle, high = triangles[0], triangles[1], triangles[2]

    def ordered_pair(swap: np.ndarray, left, right):
        return where(swap, right, left), where(swap, left, right)

    # Sort the three corners from deepest to highest and record whether the swaps flip the triangle winding.
    first_swap = first_depth < second_depth
    first_depth, second_depth = ordered_pair(first_swap, first_depth, second_depth)
    deep, middle = ordered_pair(first_swap, deep, middle)
    second_swap = second_depth < third_depth
    second_depth, third_depth = ordered_pair(second_swap, second_depth, third_depth)
    middle, high = ordered_pair(second_swap, middle, high)
    third_swap = first_depth < second_depth
    first_depth, second_depth = ordered_pair(third_swap, first_depth, second_depth)
    deep, middle = ordered_pair(third_swap, deep, middle)
    winding_flip = first_swap ^ second_swap ^ third_swap

    def water_crossing(start: Point, end: Point, start_depth: Scalar, end_depth: Scalar) -> Point:
        """The point along an edge where depth passes through zero, clamped to the segment."""
        drop = start_depth - end_depth
        share = (start_depth / where(drop.abs() > 0, drop, mv.w & start)).clip(0, 1)
        return start + share * (end - start)

    first_cut = where(second_depth < 0, water_crossing(deep, middle, first_depth, second_depth), middle)
    second_cut = where(
        second_depth < 0,
        first_cut,
        where(third_depth < 0, water_crossing(middle, high, second_depth, third_depth), high),
    )
    third_cut = where(third_depth < 0, water_crossing(deep, high, first_depth, third_depth), high)
    first_part = stack((deep, where(winding_flip, second_cut, first_cut), where(winding_flip, first_cut, second_cut)), axis=-1)
    second_part = stack((deep, where(winding_flip, third_cut, second_cut), where(winding_flip, second_cut, third_cut)), axis=-1)
    return stack((first_part, second_part), axis=-2)                # [..., 2, 3] Point-[F]


In [ ]:
def pressure_lines(triangles: Point, surface: Plane) -> tuple[Point, Forque]:
    """Quadrature sample points on the wet parts of each triangle and their inward normal lines per unit
    pressure, each weighted by one third of its wet triangle's area."""
    parts = wet_triangles(triangles, surface)                           # [..., 2, 3] Point-[F]
    samples = (parts[..., None, :] * quadrature_rule).sum(axis=-1)      # [..., 2, 3] Point-[F]
    # A part's corner join has weight equal to twice its area; multiplying by the sign of the enclosed volume
    # makes every plane face outward regardless of winding.
    volume = (origin & triangles[0] & triangles[1] & triangles[2]).sites.sum()  # [] Scalar-
    planes = (parts[..., 0] & parts[..., 1] & parts[..., 2]) * (volume / volume.norm())  # [..., 2] Plane+[F]
    return samples, -(samples | planes[..., None]) / 6                  # [..., 2, 3] Point-[F], Forque-[F]


def buoyancy(triangles: Point, surface: Plane, fluid_weight: float) -> tuple[Forque, Forque]:
    """The net hydrostatic push on the wet hull below each water surface, and its rate of growth with draft."""
    samples, lines = pressure_lines(triangles, surface)                 # [..., 2, 3] Point-[F], Forque-[F]
    depths = -(surface[..., None, None] & samples)                      # [..., 2, 3] Scalar+[F]
    push = (fluid_weight * depths * lines).sites.sum().sum(axis=(-2, -1))    # [...] Forque-
    growth = (fluid_weight * lines).sites.sum().sum(axis=(-2, -1))           # [...] Forque-
    return push, growth


topology, vertices = wigley_hull(length, beam, draft, freeboard, n_cells)   # Point-[V]
# Each triangle's corners, read off the vertices they share.
hull = topology.incidence[2, 0] * vertices                              # [3] Point-[F]
upright_push, _ = buoyancy(hull, water, fluid_weight)                   # [] Forque-
# The hull's weight is the water it displaces floating upright at its waterline.
weight = upright_push & heave                                           # [] Scalar+

print("water displaced, in cubic metres:", (weight / fluid_weight).to_array().round(4))
print("the smooth hull's, 4 / 9 of length times beam times draft:", round(4 / 9 * length * beam * draft, 4))

## Floating at a heel and the righting couple

Heeled by an angle, the body floats at the height where its buoyancy carries its weight. Newton's method finds that height, correcting a guess by the excess of weight over the upward part of the buoyancy, divided by how fast the buoyancy grows as the water surface rises, which is the water's weight times the area the surface cuts from the body.

The weight is a line of force too, straight down through the centre of gravity, `weight * (centre & -up)`. At a heel the buoyancy and the weight are parallel lines apart, and their sum is a pure couple, called the righting couple. Its size divided by the weight is the righting arm, the distance between the two lines. When the buoyancy shifts towards the side the body heels to, the couple turns it back upright.

In [ ]:
centre_heights, n_loads = scenarios.centre_heights, scenarios.n_loads
shown_load, shown_heel = scenarios.shown_load, scenarios.shown_heel
n_newton_steps = scenarios.n_newton_steps

In [ ]:
def heel_motors(angles: np.ndarray) -> Motor:
    """Motors heeling a body about the longitudinal heel axis by angles given in degrees."""
    return (heel_axis * (-angles * (np.pi / 180) / 2)).exp()


def equilibrium_heights(triangles: Point, turns: Motor, weight: Scalar, heights: Scalar, fluid_weight: float,
                        n_steps: int) -> Scalar:
    """The water-surface height over the body's origin at which buoyancy balances weight under each turn,
    refined from initial heights by Newton steps."""
    for _ in range(n_steps):
        push, growth = buoyancy(triangles, turns << (water - heights * mv.w), fluid_weight)
        upward_shift = turns << heave                                   # [...] Twist+
        heights = heights + (weight - (push & upward_shift)) / (growth & upward_shift)
    return heights                                                      # [...] Scalar+

In [ ]:
def world_placement(turns: Motor, heights: Scalar) -> Motor:
    """The motor placing the body in the world frame with the water surface at `z = 0`."""
    return (heave * (heights / 2)).exp() * turns                        # [...] Motor+


def body_forces(triangles: Point, turns: Motor, heights: Scalar, weight: Scalar, centre: Point,
                fluid_weight: float) -> tuple[Forque, Forque]:
    """The buoyancy and gravity lines acting on the floating body in its own frame, whose sum is the righting
    couple."""
    push, _ = buoyancy(triangles, turns << (water - heights * mv.w), fluid_weight)
    return push, weight * (centre & (turns << -up))                     # [...] Forque-, Forque-


centres = mv.point([[0, 0, height] for height in centre_heights])      # [n_loads] Point-
turn = heel_motors(np.array([shown_heel]))                              # [1] Motor+
height = equilibrium_heights(hull, turn, weight, unit * np.zeros(1), fluid_weight, n_newton_steps)  # [1] Scalar+
ship_push, _ = body_forces(hull, turn, height, weight, centres[shown_load], fluid_weight)  # [1] Forque-
save_figure(render.draw_ship(topology, vertices, world_placement(turn, height)[0], ship_push[0], centres[shown_load]), "floating_ship");

On the left, the hull heeled by 35°, its wet surface darker with depth. On the right, the same hull seen along its length, with the weight in orange down from the centre of gravity, the buoyancy in blue up from where its line crosses the water surface, and the righting arm between them in black. The buoyancy has moved towards the side the hull heels to, so the couple turns it back.

## Righting arms and the stiffness against tilting

For a small tilt the same sum can be taken over `Tilt`, the space of all small turns about horizontal lines. In the body's frame the water plane turns against the hull, `surface.commutator(Tilt)`, deepening each wet point in proportion to its distance from the tilt axis, while the weight line turns with it. The change of the net force is then a map from tilts to lines, and pairing it back with the tilt gives the stiffness against tilting, a quadratic form with one eigenvalue against heel and one against trim. Divided by the weight these are the metacentric heights, the slopes of the righting arm curves at upright. Large heels need the full computation, one heel at a time.

In [ ]:
heels, n_heels = scenarios.heels, scenarios.n_heels

In [ ]:
def tilt_stiffness(triangles: Point, heights: Scalar, weight: Scalar, centre: Point, fluid_weight: float) -> Stiffness:
    """The quadratic tilt stiffness of the upright body floating at each height, pairing small tilts with the
    induced change in buoyancy and weight lines."""
    surface = water - heights * mv.w                                    # [...] Plane+
    samples, lines = pressure_lines(triangles, surface)                 # [..., 2, 3] Point-[F], Forque-[F]
    # In the body's frame a small tilt turns the water plane against the hull, changing each sample's depth,
    # and turns the downward weight line with it.
    surface_rate = surface[..., None, None].commutator(Tilt)            # [..., 1, 1] Plane+ <- Tilt+
    buoyancy_change = (fluid_weight * -(surface_rate & samples) * lines).sites.sum().sum(axis=(-2, -1))  # [...] Forque- <- Tilt+
    weight_change = weight * centre.regressive((-up).commutator(Tilt))  # [...] Forque- <- Tilt+
    return Tilt & (buoyancy_change + weight_change)                     # [...] Scalar+ <- (Tilt+, Tilt+)


# Upright, the hull floats at its waterline, where the water surface passes through its origin.
stiffnesses = tilt_stiffness(hull, 0 * weight, weight, centres, fluid_weight)  # [n_loads] Scalar+ <- (Tilt+, Tilt+)
turns = heel_motors(heels)                                              # [n_heels] Motor+
# Every load weighs the same, so the hull floats at the same height at each heel, whatever the load.
heights = equilibrium_heights(hull, turns, weight, unit * np.zeros(n_heels), fluid_weight, n_newton_steps)  # [n_heels] Scalar+
push, weighing = body_forces(hull, turns[:, None], heights[:, None], weight, centres, fluid_weight)  # [n_heels, n_loads] Forque-, Forque-
couples = push + weighing                                               # [n_heels, n_loads] Forque-
save_figure(render.draw_curves(heels, couples, weight, stiffnesses), "floating_righting")

print("metacentric heights against heel and trim, in metres:\n", (stiffnesses.eigvalsh() / weight).to_array().round(3))

Each curve is the righting arm of the hull in metres over its heel from upright to upside down, with the centre of gravity at one height, from 20 cm below the waterline, darkest, to 20 cm above it, lightest. The dashed lines are the tangents at upright that the stiffness gives. With its centre of gravity low the hull rights itself from almost any heel. Raised far enough, its metacentric height turns negative, the arm is negative from the start, and the hull capsizes to float upside down, where the curves return to zero.

## The floating cube

A cube of uniform density floats with a face up only when it is light or heavy enough, below about 0.21 of the water's density or above about 0.79. In between it rests heeled, and between a quarter and three quarters of the water's density it floats edge up. The same functions, with the cube's twelve triangles as the surface, map its righting couple over density and heel; where the couple turns from pushing the cube over to pushing it back as the heel grows, the cube rests stably.

In [ ]:
densities, n_densities = scenarios.densities, scenarios.n_densities
cube_heels, n_cube_heels = scenarios.cube_heels, scenarios.n_cube_heels
shown_densities, n_shown_cubes = scenarios.shown_densities, scenarios.n_shown_cubes

In [ ]:
def stable_heels(couples: Forque, heel_grid: np.ndarray) -> Scalar:
    """Smallest non-negative heel angle in degrees at which each row of righting couples crosses upward through zero."""
    arms = -(couples & heel_axis)                                       # [..., n_cube_heels] Scalar+
    below, above = arms[..., :-1], arms[..., 1:]
    span = above - below
    # The crossing lies between two heels of the grid, in proportion to the arms there.
    share = -below / where(span.abs() > 0, span, unit)
    rests = heel_grid[:-1] + share * np.diff(heel_grid)                 # [..., n_cube_heels - 1] Scalar+
    stable = (below < 0) & (above >= 0) & (rests > -1e-6)
    first_index = np.argmax(stable, axis=-1)
    chosen = np.arange(rests.shape[-1]) == first_index[..., None]
    return (rests * chosen).sum(axis=-1)                                # [...] Scalar+


cube_topology, cube_vertices = box_surface(np.ones(3))                  # Point-[V]
cube = cube_topology.incidence[2, 0] * cube_vertices                    # [3] Point-[F]
cube_weights = unit * (fluid_weight * densities)[:, None]               # [n_densities, 1] Scalar+
cube_turns = heel_motors(cube_heels)                                    # [n_cube_heels] Motor+
cube_heights = equilibrium_heights(cube, cube_turns, cube_weights, unit * np.zeros((n_densities, n_cube_heels)),
                                   fluid_weight, n_newton_steps)        # [n_densities, n_cube_heels] Scalar+
cube_push, cube_weighing = body_forces(cube, cube_turns, cube_heights, cube_weights, origin, fluid_weight)  # [n_densities, n_cube_heels] Forque-, Forque-
cube_couples = cube_push + cube_weighing                                # [n_densities, n_cube_heels] Forque-

In [ ]:
rest_heels = stable_heels(cube_couples[np.searchsorted(densities, shown_densities)], cube_heels)  # [n_shown_cubes] Scalar+
shown_weights = unit * (fluid_weight * np.asarray(shown_densities))     # [n_shown_cubes] Scalar+
shown_turns = heel_motors(rest_heels)                                   # [n_shown_cubes] Motor+
shown_heights = equilibrium_heights(cube, shown_turns, shown_weights, unit * np.zeros(n_shown_cubes), fluid_weight,
                                    n_newton_steps)                     # [n_shown_cubes] Scalar+
cube_placements = world_placement(shown_turns, shown_heights)           # [n_shown_cubes] Motor+
save_figure(render.draw_cubes(densities, cube_heels, cube_couples, fluid_weight, cube_topology, cube_vertices,
                              cube_placements, shown_densities, rest_heels), "floating_cubes");

The map shows the righting couple over the cube's density, across, and its heel from upright to a quarter turn, up, blue where it turns the cube back towards upright and red where it pushes the cube further over. Where red below meets blue above, the cube rests stably, drawn thick; the other way round it balances unstably, drawn thin. The cubes above rest at the white dots, from face up through tilted to edge up and back.

## Rolling after release from a heel

The hull is released from a heel of 30°, once with its centre of gravity 10 cm below the waterline and once 20 cm above it. It turns about its heel axis, floating at each moment at the height where its buoyancy carries its weight. The righting couple paired with the heel axis, divided by the hull's moment of inertia about that axis, speeds its roll up or slows it down, and a little damping takes energy out. The roll and its rate are twists, the heel axis scaled by the angle and by the rate. The cell that releases the hull takes about ten seconds, most of it drawing the frames.

In [ ]:
released_loads, release_heel = scenarios.released_loads, scenarios.release_heel
gyration, damping = scenarios.gyration, scenarios.damping
step_dt, n_frames, n_steps_per_frame = scenarios.step_dt, scenarios.n_frames, scenarios.n_steps_per_frame
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def release_states(hull: Point, weight: Scalar, centres: Point) -> Iterator[tuple[Twist, Motor, Forque, Forque]]:
    """Yield roll twist, world placement, buoyancy and weight as the heeled hull rolls under hydrostatic torque."""
    n_cases = centres.shape[0]
    inertia = weight / gravity * (gyration * beam) ** 2                 # [] Scalar+
    roll = heel_axis * np.full(n_cases, np.radians(release_heel))       # [n_cases] Twist+
    roll_rate = heel_axis * np.zeros(n_cases)                           # [n_cases] Twist+
    turns = (roll * (-1 / 2)).exp()                                     # [n_cases] Motor+
    heights = equilibrium_heights(hull, turns, weight, unit * np.zeros(n_cases), fluid_weight, n_newton_steps)  # [n_cases] Scalar+
    push, weighing = body_forces(hull, turns, heights, weight, centres, fluid_weight)  # [n_cases] Forque-, Forque-
    for _ in range(n_frames):
        yield roll, world_placement(turns, heights), push, weighing
        for _ in range(n_steps_per_frame):
            # Roll acceleration comes from the righting couple paired with the heel axis, damped linearly.
            roll_rate = roll_rate + (heel_axis * (((push + weighing) & heel_axis) / inertia) - damping * roll_rate) * step_dt
            roll = roll + roll_rate * step_dt
            turns = (roll * (-1 / 2)).exp()
            # The previous height is a close start, so one Newton step keeps the hull afloat.
            heights = equilibrium_heights(hull, turns, weight, heights, fluid_weight, 1)
            push, weighing = body_forces(hull, turns, heights, weight, centres, fluid_weight)


roll_history = show_states(release_states(hull, weight, centres[released_loads]), render.animate_release,
                           topology, vertices, centres[released_loads], heels, couples[:, released_loads], weight,
                           name="floating_release", duration_ms=frame_duration_ms)

With its centre of gravity low the hull rocks back and settles upright; with it high the hull rolls over and settles upside down. On the right, each load's righting arm curve, continued past half a turn, carries the hull's heel and arm so far.

In [ ]:
# checks
# The slope of each righting arm curve at upright is the metacentric height against heel.
arms = -(couples & heel_axis) / weight                                  # [n_heels, n_loads] Scalar+
metacentric_heights = stiffnesses(heel_axis, heel_axis) / weight        # [n_loads] Scalar+
assert ((arms[1] / np.sin(np.radians(heels[1])) - metacentric_heights).abs() < 1e-2).all()
# A cube floats face up only below a density of (3 - sqrt(3)) / 6 or above (3 + sqrt(3)) / 6.
face_up = stable_heels(cube_couples, cube_heels).abs() < 1e-6
assert np.array_equal(face_up, (densities < (3 - np.sqrt(3)) / 6) | (densities > (3 + np.sqrt(3)) / 6))
# The hull with the low load comes back upright; with the high one it capsizes and floats upside down.
final_rolls = (roll_history[-1][0] & heel_axis.dual()) * (180 / np.pi)  # [2] Scalar+
assert (final_rolls[0].abs() < 5).all() and ((final_rolls[1] - 180).abs() < 10).all()